# Scheduling Algorithms
Autor: Prof. Lucas Nunes Alegre

In [59]:
import heapq
import time
from collections import deque
from itertools import permutations

## Escalonamento de Intervalos

In [ ]:
# Cada intervalo tem um tempo de início e um tempo de fim
interv_a = [(0, 9), (1, 2), (3, 4), (5, 6), (7, 8)]
interv_b = [(0, 5), (6, 10), (4, 7)]
interv_c = [(0, 2), (1, 4), (1, 4), (1, 4), (3, 6), (5, 8), (7, 10), (9, 12), (9, 12), (9, 12), (11, 14)]

In [ ]:
def termina_mais_cedo(intervalos):
    """Algoritmo Guloso para Escalonamento de Intervalos.
    O(n log n)
    """
    intervalos = sorted(intervalos, key=lambda x: x[1])  # ordena lista de intervalos pelo segundo componente (fim)
    ordem = [intervalos.pop(0)]  # lista de intervalos selecionados, inicializada com o primeiro intervalo

    for intervalo in intervalos:
        if intervalo[0] > ordem[0][1]:  # se o intervalo é compatível com o último inserido na solução
            ordem.append(intervalo)   # insere intervalo na solução

    return ordem

In [52]:
termina_mais_cedo(interv_c)

[(0, 2), (3, 6), (5, 8), (7, 10), (9, 12), (9, 12), (9, 12), (11, 14)]

## Particionamento de intervalos

In [ ]:
# Cada intervalo tem um tempo de início e um tempo de fim
interv_d = [(0, 11), (0, 1), (2, 5), (7, 10), (0, 3), (4, 6), (8, 12), (13, 16), (14, 15)]

In [ ]:
def inicia_mais_cedo(intervalos):
    """Algoritmo Guloso para Particionamento de Intervalos
        O(n^2)
    """
    intervalos = sorted(intervalos, key=lambda x: x[0])
    num_salas = 0
    salas = {}   # mapeia número da sala para lista de intervalos
    for intervalo in intervalos:

        achou = False
        for sala, intervalos_na_sala in salas.items():
            ultima_sala = intervalos_na_sala[-1]
            if intervalo[0] >= ultima_sala[1]:  # intervalo começa depois que o último intervalo na sala termina
                achou = True
                salas[sala].append(intervalo)
                break
        
        if not achou: # intervalo não é compatível com nenhum sala existente
            num_salas += 1
            salas[num_salas] = [intervalo]  # cria nova sala
    
    return salas

In [ ]:
def inicia_mais_cedo_heap(intervalos):
	"""Algoritmo Guloso para Particionamento de Intervalos - Versão usando Heap
        O(n log n)
    """
	intervalos = sorted(intervalos, key=lambda x: x[0]) # ordena lista de intervalos pelo primeiro componente (inicio)
	
	aloca = []  # heap contendo tripla (tempo-liberacao, nro sala, lista de tarefas). Chave = tempo-liberacao
	
	h  = intervalos.pop(0)  # extrai primeiro intervalo h
	heapq.heappush(aloca, (h[1], 1, [h]))  # inicializa a heap com ele

	for x in intervalos:
		(f, s, l) = aloca[0]    # acessa o primeiro elemento da heap (elemento de f mínimo), sem remover

		if x[0] <= f:  # se o início de x conflita com o término mínimo de salas, não há sala disponível
			n = len(aloca) + 1    # cria novo nome de sala
			heapq.heappush(aloca, (x[1], n, [x]))  # insere x na nova sala (na heap)

		else:  # se não há conflito, atualizar sala s na heap, incluindo x e atualizando f como 
			heapq.heappop(aloca)              # remove o primeiro elemento da lista
			l.insert(0, x)                     # adiciona x à lista de s
			heapq.heappush(aloca,(x[1], s, l))  # inclui a atualização na heap

	return aloca

In [48]:
inicia_mais_cedo(interv_d)

{1: [(0, 11), (13, 16)],
 2: [(0, 1), (2, 5), (7, 10), (14, 15)],
 3: [(0, 3), (4, 6), (8, 12)]}

In [49]:
inicia_mais_cedo_heap(interv_d)

[(12, 3, [(8, 12), (4, 6), (0, 3)]),
 (16, 2, [(13, 16), (7, 10), (2, 5), (0, 1)]),
 (15, 1, [(14, 15), (0, 11)])]

## Minimização de Atraso Máximo

In [24]:
# cada tarefa tem um nome, duração e um deadline
tarefas = [('a', 3, 6), ('b', 2, 8), ('c', 1, 9), ('d', 4, 9), ('e', 3, 14), ('f', 2, 15)]

In [60]:
def minimiza_atraso(tarefas):
    """Algoritmo Guloso para Minimizar Atraso Máximo.
    O(n log n)
    """
    tarefas = sorted(tarefas, key=lambda x: x[2]) # ordena em ordem crescente de deadline
    ordem = []
    t = 0
    maior_delay = 0
    for tarefa in tarefas:
        fim = t + tarefa[1]
        ordem.append((tarefa[0], t, fim))
        delay = max(fim - tarefa[2], 0)
        maior_delay = max(delay, maior_delay)
        print(f"Tarefa {tarefa[0]} - inicio: {t} fim: {fim} delay: {delay}")
        t = fim
    
    print(f"Maior Delay: {maior_delay}")

    return ordem

In [61]:
minimiza_atraso(tarefas)

Tarefa a - inicio: 0 fim: 3 delay: 0
Tarefa b - inicio: 3 fim: 5 delay: 0
Tarefa c - inicio: 5 fim: 6 delay: 0
Tarefa d - inicio: 6 fim: 10 delay: 1
Tarefa e - inicio: 10 fim: 13 delay: 0
Tarefa f - inicio: 13 fim: 15 delay: 0
Maior Delay: 1


[('a', 0, 3),
 ('b', 3, 5),
 ('c', 5, 6),
 ('d', 6, 10),
 ('e', 10, 13),
 ('f', 13, 15)]

### Caching Ótimo

In [55]:
fila = ["A", "B", "C", "B", "C", "A"]

In [ ]:
def farthest_in_future(fila, tamanho_cache=2):
    """Algoritmo Guloso para Minimizar Cache Miss
    O(n * k), onde n é o tamanho da fila e k o tamanho da cache.
    """

    tempos_tarefa = {}  # Mapeia cada tarefa para uma lista de índices onde ela aparece na fila
    for i in range(len(fila)):
        if fila[i] in tempos_tarefa:
            tempos_tarefa[fila[i]].append(i)
        else:
            tempos_tarefa[fila[i]] = deque([i])

    cache = []
    misses = 0
    for i, tarefa in enumerate(fila):
        tempos_tarefa[tarefa].popleft()  # Retira o índice da aparição da tarefa

        if tarefa in cache:  # Cache Hit
            continue
        else:                # Cache Miss
            misses += 1

            if len(cache) < tamanho_cache:  # Se tem espaço no cache, inclue a tarefa
                cache.append(tarefa)

            else:  # Senão, remove a tarefa cujo próximo índice na fila é maior (tarefa mais longe/farthest in future)
                tarefa_mais_longe = -1
                tempo_mais_longe = -1
                for t in cache:
                    proxima_aparicao = tempos_tarefa[t][0] if len(tempos_tarefa[t]) > 0 else -1
                    if proxima_aparicao > tempo_mais_longe:
                        tarefa_mais_longe = t
                        tempo_mais_longe = proxima_aparicao
                # print(tarefa_mais_longe, fila[i:], tempos_tarefa)
                if tarefa_mais_longe != -1:
                    cache.remove(tarefa_mais_longe)  # remove do cache a tarefa que vai demorar mais a aparecer
                    cache.append(tarefa)
    
    return misses

In [83]:
farthest_in_future(fila)

4